# 🎶 YuE2-Studio (Colab & Kaggle Edition)
Dashboard generasi musik **YuE2** + **Cover Mode (SheetSage2)** + **LoRA Training Studio** untuk **Google Colab** dan **Kaggle** (kompatibel dengan GPU **T4 16 GB**).

- **Unified Engine & LoRA Trainer:** `timoncool/HOT-Step-CPP` (`ace-server` + `ace-train` CUDA)
- **Models:** `scragnog/YuE2-GGUF` (`yue2-lm-q8_0.gguf`, `yue2-vae-standard-f32.gguf`, `yue2-tok-f16.gguf`)
- **Dashboard:** Gradio Bilingual (ID/EN) dengan dukungan LoRA Mixer (`lm_adapter`)

In [ ]:
#@title 1. Deteksi Platform (Colab / Kaggle), Cek GPU & Mount Google Drive
USE_GOOGLE_DRIVE = True #@param {type:"boolean"}
DRIVE_FOLDER = "/content/drive/MyDrive/YuE2-Studio" #@param {type:"string"}

import os, pathlib
!nvidia-smi

IS_KAGGLE = pathlib.Path("/kaggle/working").exists()
IS_COLAB = pathlib.Path("/content").exists() and not IS_KAGGLE
BASE_ROOT = pathlib.Path("/kaggle/working") if IS_KAGGLE else pathlib.Path("/content")
WORKSPACE = BASE_ROOT / "yue2-workspace"

for sub in ["models", "adapters", "library", "datasets", "trained_loras", "bin"]:
    (WORKSPACE / sub).mkdir(parents=True, exist_ok=True)

if IS_KAGGLE:
    USE_GOOGLE_DRIVE = False
    print(f"🟢 Terdeteksi: KAGGLE | Workspace: {WORKSPACE}")
elif IS_COLAB and USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    drive_root = pathlib.Path(DRIVE_FOLDER)
    for sub in ["bin", "models", "adapters", "library", "datasets", "trained_loras"]:
        (drive_root / sub).mkdir(parents=True, exist_ok=True)
    print(f"🟢 Terdeteksi: GOOGLE COLAB | Drive Cache: {drive_root}")
else:
    print(f"ℹ️ Workspace lokal: {WORKSPACE}")

os.environ["YUE_WORKSPACE"] = str(WORKSPACE)
os.environ["YUE_DRIVE_ROOT"] = DRIVE_FOLDER

In [ ]:
#@title 2. Install Dependensi & Build HOT-Step-CPP (ace-server + ace-train CUDA)
import os, pathlib, shutil, subprocess

SERVER_BIN = WORKSPACE / "bin/ace-server"
TRAIN_BIN = WORKSPACE / "bin/ace-train"
DRIVE_BIN_DIR = pathlib.Path(DRIVE_FOLDER) / "bin"

if USE_GOOGLE_DRIVE and (DRIVE_BIN_DIR / "ace-server").exists() and (DRIVE_BIN_DIR / "ace-train").exists() and not SERVER_BIN.exists():
    print("📦 Memulihkan ace-server, ace-train & shared libs dari cache Google Drive...")
    !cp -a "{DRIVE_BIN_DIR}"/* "{WORKSPACE / 'bin'}/" 2>/dev/null || true
    SERVER_BIN.chmod(0o755)
    TRAIN_BIN.chmod(0o755)

if not SERVER_BIN.exists() or not TRAIN_BIN.exists():
    !apt-get update -qq && apt-get install -y -qq cmake ninja-build libcurl4-openssl-dev ffmpeg
    try:
        cap = subprocess.check_output(["nvidia-smi", "--query-gpu=compute_cap", "--format=csv,noheader"], text=True).strip().splitlines()[0]
        GPU_ARCH = cap.replace(".", "")
    except Exception:
        GPU_ARCH = "75"
    print(f"🎯 Target CUDA Architecture: sm_{GPU_ARCH}")
    SRC_HOT = BASE_ROOT / "HOT-Step-CPP"
    if not SRC_HOT.exists():
        !git clone --recurse-submodules https://github.com/timoncool/HOT-Step-CPP.git "{SRC_HOT}"
    !rm -rf "{SRC_HOT}/engine/build"
    !cmake -S "{SRC_HOT}/engine" -B "{SRC_HOT}/engine/build" -G Ninja -DGGML_CUDA=ON -DCMAKE_CUDA_COMPILER=/usr/local/cuda/bin/nvcc -DCMAKE_CUDA_ARCHITECTURES="{GPU_ARCH}"
    !cmake --build "{SRC_HOT}/engine/build" --target ace-server ace-train --config Release -j $(nproc)
    for tname, dest in [("ace-server", SERVER_BIN), ("ace-train", TRAIN_BIN)]:
        built = next((p for p in [SRC_HOT / f"engine/build/{tname}", SRC_HOT / f"engine/build/bin/{tname}"] if p.exists()), None)
        if not built:
            raise FileNotFoundError(f"Binary {tname} tidak ditemukan setelah build.")
        shutil.copy2(built, dest)
        dest.chmod(0o755)
    !find "{SRC_HOT}/engine/build" -name '*.so*' -exec cp -a {{}} "{WORKSPACE / 'bin'}/" \;
    if USE_GOOGLE_DRIVE:
        !cp -a "{WORKSPACE / 'bin'}"/* "{DRIVE_BIN_DIR}/"
        print("💾 ace-server, ace-train & libs disimpan ke Google Drive cache.")

os.environ["YUE_SERVER_BIN"] = str(SERVER_BIN)
os.environ["ACE_TRAIN_BIN"] = str(TRAIN_BIN)
print(f"✅ Engine siap: {SERVER_BIN} & {TRAIN_BIN}")

In [ ]:
#@title 3. Download Model YuE2 (scragnog/YuE2-GGUF — untuk Inference & LoRA Training)
QUANT = "q8_0" #@param ["q8_0", "Q6_K", "Q5_K_M", "Q4_K_M", "bf16"]
!pip install -q gradio>=4.44.0 requests>=2.31.0 huggingface_hub>=0.25.0 pydantic>=2.7.0

import pathlib, shutil
from huggingface_hub import hf_hub_download

MODELS_DIR = WORKSPACE / "models"
DRIVE_MODELS = pathlib.Path(DRIVE_FOLDER) / "models"

for fname in [f"yue2-lm-{QUANT}.gguf", "yue2-vae-standard-f32.gguf", "yue2-tok-f16.gguf"]:
    local_f = MODELS_DIR / fname
    drive_f = DRIVE_MODELS / fname
    if USE_GOOGLE_DRIVE and drive_f.exists() and not local_f.exists():
        print(f"📦 Menyalin {fname} dari Google Drive...")
        shutil.copy2(drive_f, local_f)
    elif not local_f.exists():
        print(f"⬇️ Mengunduh {fname} dari scragnog/YuE2-GGUF...")
        hf_hub_download(repo_id="scragnog/YuE2-GGUF", filename=fname, local_dir=str(MODELS_DIR))
        if USE_GOOGLE_DRIVE:
            shutil.copy2(local_f, drive_f)
    print(f"✅ {fname} siap.")

os.environ["YUE_MODELS_DIR"] = str(MODELS_DIR)

In [ ]:
#@title 4. Jalankan ace-server (LoRA-ready) & YuE2-Studio Dashboard (Gradio)
GITHUB_REPO = "https://github.com/USERNAME/yuestudio.git" #@param {type:"string"}
import os, pathlib, subprocess, time, requests

REPO_DIR = BASE_ROOT / "yuestudio"
if not (REPO_DIR / "dashboard/app.py").exists():
    drive_repo = pathlib.Path(DRIVE_FOLDER) / "yuestudio"
    kaggle_in = pathlib.Path("/kaggle/input/yuestudio")
    if GITHUB_REPO and "USERNAME" not in GITHUB_REPO:
        print(f"⬇️ Cloning repo dari GitHub: {GITHUB_REPO}")
        !git clone --depth 1 "{GITHUB_REPO}" "{REPO_DIR}"
    elif USE_GOOGLE_DRIVE and (drive_repo / "dashboard/app.py").exists():
        !cp -r "{drive_repo}" "{REPO_DIR}"
    elif (kaggle_in / "dashboard/app.py").exists():
        !cp -r "{kaggle_in}" "{REPO_DIR}"
    else:
        raise FileNotFoundError(f"Isi GITHUB_REPO di atas dengan URL repo GitHub kamu, atau upload folder 'dashboard/' ke {REPO_DIR}")

!pkill -f yue-server || true
!pkill -f ace-server || true
server_cmd = [
    str(WORKSPACE / "bin/ace-server"),
    "--models", str(WORKSPACE / "models"),
    "--adapters", str(WORKSPACE / "adapters"),
    "--host", "0.0.0.0",
    "--port", "8087",
]
os.environ["LD_LIBRARY_PATH"] = f"{WORKSPACE / 'bin'}:{os.environ.get('LD_LIBRARY_PATH', '')}"
log_f = open(str(BASE_ROOT / "yue-server.log"), "w")
server_proc = subprocess.Popen(server_cmd, stdout=log_f, stderr=subprocess.STDOUT, env=os.environ.copy())

print("⏳ Menunggu ace-server aktif di port 8087...")
server_ready = False
for _ in range(45):
    if server_proc.poll() is not None:
        print("❌ ace-server berhenti dengan error! Isi yue-server.log:")
        !tail -n 30 "{BASE_ROOT / 'yue-server.log'}"
        break
    try:
        if requests.get("http://127.0.0.1:8087/health", timeout=2).status_code == 200:
            print("🟢 ace-server ONLINE (YuE2 + LoRA support aktif)!")
            server_ready = True
            break
    except Exception:
        pass
    time.sleep(2)
if not server_ready and server_proc.poll() is None:
    print("⚠️ ace-server masih loading, cek log di bawah:")
    !tail -n 20 "{BASE_ROOT / 'yue-server.log'}"

os.environ["GRADIO_SHARE"] = "1"
%cd "{REPO_DIR}"
!python -m dashboard.app

In [ ]:
#@title 5. (Opsional) Tunnel WebUI Bawaan HOT-Step-CPP (Port 8087 via Cloudflared)
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared
!cloudflared tunnel --url http://127.0.0.1:8087